# 02 — Two-Qubit Encoding and Pauli Form

Map the 3-state momentum basis onto the 4 computational basis states of two qubits, verify the padded 4x4 Hamiltonian reproduces the exact 3-state physics, and rewrite it as a sum of Pauli strings for use as a VQE operator.

Encoding:
* $|-1\rangle \to |00\rangle$
* $|0\rangle \to |01\rangle$
* $|{+1}\rangle \to |10\rangle$
* $|11\rangle$ unused — given a penalty energy so VQE avoids it

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import numpy as np
import pandas as pd

from src.hamiltonian import bragg_hamiltonian
from src.encoding import padded_bragg_hamiltonian, pauli_decompose

## Verify the padded Hamiltonian reproduces the 3-state physics

In [2]:
delta, omega, penalty = 1.0, 0.5, 13.0

H3 = bragg_hamiltonian(delta=delta, omega=omega)
H4 = padded_bragg_hamiltonian(delta=delta, omega=omega, penalty=penalty)

e3 = np.linalg.eigvalsh(H3)
e4 = np.linalg.eigvalsh(H4)

print("3-state eigenvalues:", e3)
print("4-state eigenvalues (lowest 3):", np.sort(e4)[:3])
print("Physical spectrum preserved:", np.allclose(np.sort(e3), np.sort(e4)[:3]))

3-state eigenvalues: [-0.12865236  3.07793022  5.05072214]
4-state eigenvalues (lowest 3): [-0.12865236  3.07793022  5.05072214]
Physical spectrum preserved: True


## Pauli decomposition

Check that the Pauli-form operator matches the padded matrix exactly.

In [3]:
delta_pauli, omega_pauli, penalty_pauli = 0.0, 0.5, 13.0

H_matrix_pauli = padded_bragg_hamiltonian(delta_pauli, omega_pauli, penalty_pauli)
H_pauli_op = pauli_decompose(H_matrix_pauli)

print(f"delta={delta_pauli}, Omega={omega_pauli}, penalty={penalty_pauli}")
print("Matches padded Hamiltonian exactly:", np.allclose(H_pauli_op.to_matrix(), H_matrix_pauli))

pauli_table = pd.DataFrame({
    "Pauli string": H_pauli_op.paulis.to_labels(),
    "coefficient": H_pauli_op.coeffs.real,
}).sort_values("Pauli string").reset_index(drop=True)
pauli_table

delta=0.0, Omega=0.5, penalty=13.0
Matches padded Hamiltonian exactly: True


,Pauli string,coefficient
0,II,5.25
1,IX,0.25
2,IZ,-1.25
3,XX,0.25
4,YY,0.25
5,ZI,-3.25
6,ZX,0.25
7,ZZ,3.25
